# CPV301 D1 Pilot R4 epoch-0100 packaging repair

Resume-only post-train repair for the verified `epoch-0100` checkpoint. It never trains another epoch. The governed training checkout, R3 predecessor, and independently QA-passed R4 sidecar are fetched with separate immutable commit and SHA-256 pins. Run all once on L4. Sealed-test export and final evaluation are forbidden.


In [ ]:
from google.colab import drive

drive.mount("/content/drive")

In [ ]:
import hashlib
import os
import re
import stat
from pathlib import Path

MODE = "resume"  # fixed: package verified epoch-0100; never train again
TRAINING_COMMIT = "939f6bbce9f5c955b13ebcc84addd5083d242355"  # checkpoint-bound clean training checkout
CODE_COMMIT = TRAINING_COMMIT
RECOVERY_COMMIT = "e20a69541e3a6327739238fc71691109ce8f51d0"  # hash-pinned R3 predecessor sidecar
RECOVERY_RUNNER_SHA256 = "8c9b23472091f6c677d635a54e4f4e8b896e1fcf204d627ba56eb208db00c49c"
RECOVERY_RUNNER = Path("/content/run_d1_colab_pilot_r3.py")
RECOVERY_RUNNER_URL = (
    "https://raw.githubusercontent.com/ScottPhantom/Traffic_SIgn_Detection/"
    f"{RECOVERY_COMMIT}/detector/colab/run_d1_colab_pilot_r3.py"
)
R4_COMMIT = "2cc45b467456e9300df613839d94c5cb0bbdcbba"  # independently QA-passed R4 packaging repair
R4_RUNNER_SHA256 = "ffcc88e2339b349775ca34e5e30531a102d0bcab1d45fa0dff9b1ea8bead25ad"
R4_RUNNER = Path("/content/run_d1_colab_pilot_r4.py")
R4_RUNNER_URL = (
    "https://raw.githubusercontent.com/ScottPhantom/Traffic_SIgn_Detection/"
    f"{R4_COMMIT}/detector/colab/run_d1_colab_pilot_r4.py"
)
RUN_LABEL = "pilot_seed_20261018_r2"
DRIVE_ROOT = Path("/content/drive/MyDrive/CPV301/Traffic_Sign_Detector/M1_D1_Colab")
DRIVE_BUNDLE = DRIVE_ROOT / "m1_colab_train_val_backend_r1.tar.gz"
SIDECAR = DRIVE_ROOT / "m1_colab_train_val_backend_r1.tar.gz.manifest.json"
STATE_DIR = DRIVE_ROOT / "runs" / RUN_LABEL
REPO_DIR = Path("/content/Traffic_Sign_Detector")
REPO_URL = "https://github.com/ScottPhantom/Traffic_SIgn_Detection.git"
EXPECTED_BUNDLE_BYTES = 766226800
EXPECTED_BUNDLE_SHA256 = "7ea344cb1e2c166786d0552f215993969efec9b5f06a9bf49310d4390243d641"
EXPECTED_SIDECAR_BYTES = 1499866
EXPECTED_SIDECAR_SHA256 = "7e0dc5b0144faa5f0be407224eaf8070dd51199fa8a1d2f164b248fd6a06d5b2"
PART_SPECS = (
    (
        "m1_colab_train_val_backend_r1.tar.gz.part-00",
        99614720,
        "53f30f08406fbef5b8451610b2fd44a1f2be42ca6d50f612b68e5ace285e66ff",
    ),
    (
        "m1_colab_train_val_backend_r1.tar.gz.part-01",
        99614720,
        "8451fe932760179d4b2e1a9cbdd21fd58744bc6b8f3d2a90e431cece7f546544",
    ),
    (
        "m1_colab_train_val_backend_r1.tar.gz.part-02",
        99614720,
        "867c935bf152ddfc3e7919547dccf3682291326b60b956a8100fc1a59b60e98d",
    ),
    (
        "m1_colab_train_val_backend_r1.tar.gz.part-03",
        99614720,
        "046fc8ae5d4e0a91196ccd6a6afa422fb1254baea7ae30bc3cab39602b534907",
    ),
    (
        "m1_colab_train_val_backend_r1.tar.gz.part-04",
        99614720,
        "6d3751d858232fada5d57e8e892f3e272cd5b9e34efb184971b3f4786842665d",
    ),
    (
        "m1_colab_train_val_backend_r1.tar.gz.part-05",
        99614720,
        "938106e4d34743e9b79109feb98d573f317afad44289f792f070f22e3924159b",
    ),
    (
        "m1_colab_train_val_backend_r1.tar.gz.part-06",
        99614720,
        "3497c1a10314c74d4b81ba9a5a49653b16ba2e2d52b424fd7655623547cb6408",
    ),
    (
        "m1_colab_train_val_backend_r1.tar.gz.part-07",
        68923760,
        "5f5ca85bc392310b2b5693ad9d0723f046eca9fd7980b45c944f22a3b3bc0eea",
    ),
)
assert MODE == "resume"
assert re.fullmatch(r"[0-9a-f]{40}", CODE_COMMIT), "Wait for QA, commit and push before Colab"


def sha256_file(path):
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()

def verified_runner_bytes(path, expected_sha256):
    fd = os.open(path, os.O_RDONLY | os.O_NOFOLLOW)
    try:
        info = os.fstat(fd)
        assert stat.S_ISREG(info.st_mode), "Runner is not a regular file"
        chunks = []
        while True:
            chunk = os.read(fd, 1024 * 1024)
            if not chunk:
                break
            chunks.append(chunk)
    finally:
        os.close(fd)
    payload = b"".join(chunks)
    assert hashlib.sha256(payload).hexdigest() == expected_sha256
    return payload


In [ ]:
import json
assert SIDECAR.is_file() and SIDECAR.stat().st_size == EXPECTED_SIDECAR_BYTES
assert sha256_file(SIDECAR) == EXPECTED_SIDECAR_SHA256
if DRIVE_BUNDLE.is_file():
    BUNDLE = DRIVE_BUNDLE
    bundle_source = "single_archive"
else:
    parts = []
    for name, expected_bytes, expected_sha256 in PART_SPECS:
        part = DRIVE_ROOT / name
        assert part.is_file() and part.stat().st_size == expected_bytes, part
        assert sha256_file(part) == expected_sha256, part
        parts.append(part)
    BUNDLE = Path("/content/m1_colab_train_val_backend_r1.tar.gz")
    if not BUNDLE.exists():
        with BUNDLE.open("xb") as destination:
            for part in parts:
                with part.open("rb") as source:
                    for chunk in iter(lambda: source.read(1024 * 1024), b""):
                        destination.write(chunk)
    bundle_source = "verified_drive_parts"
assert BUNDLE.stat().st_size == EXPECTED_BUNDLE_BYTES
assert sha256_file(BUNDLE) == EXPECTED_BUNDLE_SHA256
assert (STATE_DIR / "recovery/latest.json").is_file(), "Verified recovery pointer missing"
preflight_latest = json.loads((STATE_DIR / "recovery/latest.json").read_text())
assert preflight_latest["generation"] == "epoch-0100", preflight_latest
print({"mounted_bundle_preflight": "PASS", "source": bundle_source, "mode": MODE})

In [ ]:
import shutil
import subprocess
import sys
import urllib.request

if REPO_DIR.exists():
    shutil.rmtree(REPO_DIR)
subprocess.run(["git", "clone", "--filter=blob:none", REPO_URL, str(REPO_DIR)], check=True)
subprocess.run(["git", "checkout", CODE_COMMIT], cwd=REPO_DIR, check=True)
observed = subprocess.run(
    ["git", "rev-parse", "HEAD"], cwd=REPO_DIR, check=True, capture_output=True, text=True
).stdout.strip()
assert observed == TRAINING_COMMIT
def download_verified_runner(path, url, expected_sha256):
    payload = urllib.request.urlopen(url, timeout=60).read()
    assert hashlib.sha256(payload).hexdigest() == expected_sha256
    if os.path.lexists(path):
        assert verified_runner_bytes(path, expected_sha256) == payload
    else:
        flags = os.O_WRONLY | os.O_CREAT | os.O_EXCL | os.O_NOFOLLOW
        fd = os.open(path, flags, 0o600)
        with os.fdopen(fd, "wb") as handle:
            handle.write(payload)
        assert verified_runner_bytes(path, expected_sha256) == payload
    return payload

r3_bytes = download_verified_runner(
    RECOVERY_RUNNER, RECOVERY_RUNNER_URL, RECOVERY_RUNNER_SHA256
)
r4_bytes = download_verified_runner(R4_RUNNER, R4_RUNNER_URL, R4_RUNNER_SHA256)
assert not subprocess.run(
    ["git", "status", "--porcelain"], cwd=REPO_DIR, check=True, capture_output=True, text=True
).stdout.strip()
if shutil.which("uv") is None:
    subprocess.run([sys.executable, "-m", "pip", "install", "uv"], check=True)
print({"training_commit": observed, "r3_commit": RECOVERY_COMMIT, "r3_sha256": RECOVERY_RUNNER_SHA256, "r4_commit": R4_COMMIT, "r4_sha256": R4_RUNNER_SHA256, "checkout_clean": True})

In [ ]:
verified_r4_bytes = verified_runner_bytes(R4_RUNNER, R4_RUNNER_SHA256)
command = [
    sys.executable,
    "-",
    "--repo-root",
    str(REPO_DIR),
    "--bundle",
    str(BUNDLE),
    "--state-dir",
    str(STATE_DIR),
    "--mode",
    MODE,
    "--seed",
    "20261018",
    "--epochs",
    "100",
    "--imgsz",
    "1280",
    "--batch",
    "4",
    "--patience",
    "0",
    "--device",
    "0",
    "--tag",
    "pilot_100e_seed_20261018_r2",
    "--download-approved-weight",
]
print("Running:", " ".join(command))
subprocess.run(command, input=verified_r4_bytes, check=True)

In [ ]:
import importlib.util
import json
import sys

execution = json.loads((STATE_DIR / "colab_execution.json").read_text())
latest = json.loads((STATE_DIR / "recovery/latest.json").read_text())
assert execution["sealed_test_accessed"] is False
assert execution["status"] == "SUCCEEDED", execution
archives = sorted(STATE_DIR.glob("*_result.tar.gz"))
assert len(archives) == 1, archives
assert execution["training_outcome"]["status"] in {
    "COMPLETED", "COMPLETED_FROM_FINAL_CHECKPOINT"
}
assert execution["training_outcome"]["completed_epoch"] == 100
assert latest["generation"] == "epoch-0100", latest
spec = importlib.util.spec_from_file_location("d1_pilot_r4_live_verifier", R4_RUNNER)
assert spec is not None and spec.loader is not None
verifier = importlib.util.module_from_spec(spec)
sys.modules[spec.name] = verifier
spec.loader.exec_module(verifier)
pilot_r3 = verifier.load_pilot_r3(REPO_DIR)
runtime_evidence = pilot_r3.verify_live_state(STATE_DIR, REPO_DIR)
assert runtime_evidence["runtime_evidence"]["distinct_runtime_digests"] is True
assert runtime_evidence["runtime_evidence"]["sealed_test_accessed"] is False
print({
    "pilot_100_epochs_and_packaging": "PASS",
    "archive": str(archives[0]),
    "latest": latest,
    "runtime_evidence": runtime_evidence,
})
